# Compute Market Qwen3 GRPO

<a href="https://colab.research.google.com/github/kiankyars/lambdatheta/blob/main/training/Compute_Market_Qwen3_GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Minimal Colab notebook for GRPO on the OpenEnv compute market environment.
Use this first for a smoke run before trying the larger gpt-oss notebook.

In [ ]:
%%capture
!pip install --upgrade uv
!uv pip install unsloth vllm --torch-backend=auto
!uv pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo
!uv pip install transformers==4.56.2 'trl>=0.24.0' datasets openenv-core
!git clone https://github.com/kiankyars/lambdatheta.git
!pip install git+https://huggingface.co/spaces/openenv-community/compute_market_env


In [ ]:
import os, sys
os.environ['OPENENV_URL'] = 'https://openenv-community-compute-market-env.hf.space'
sys.path.append('/content/lambdatheta')
MAX_STEPS = 300
LORA_RANK = 32
MODEL_NAME = 'unsloth/Qwen3-4B-Base'


In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from training.compute_market_grpo import build_trainer, build_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
dataset = build_dataset(size=64)
trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    env_url=os.environ['OPENENV_URL'],
    train_dataset=dataset,
    output_dir='outputs/compute-market-qwen3-4b',
    max_steps=MAX_STEPS,
)
trainer


In [ ]:
trainer.train()
